# 02 — Histogram, Kontras, dan Normalisasi

### Dari distribusi intensitas ke keputusan pengukuran

**Tujuan utama:** membedakan *citra yang lebih mudah dilihat* dari *citra yang lebih akurat secara radiometrik*. Histogram adalah ringkasan statistik satu dimensi dari citra dua dimensi. Kita akan mengkaji kekuatan, kehilangan informasi, dan dampaknya terhadap inferensi.

**Prasyarat:** Notebook 01 (radiometri, sampling, kuantisasi, noise sensor); NumPy dan probabilitas dasar. **Waktu:** 2–3 sesi kuliah/praktikum; studi kasus dapat dijadikan proyek mandiri.

> **Status data:** citra `camera` dan `astronaut` adalah contoh bawaan `scikit-image`; *phantom* dan pengukuran sensor pada studi kasus adalah **simulasi sintetis**, bukan data eksperimen dari kamera riil. Semua eksperimen dijalankan lokal setelah dependensi tersedia.

## Peta kompetensi dan pertanyaan pemandu

Setelah menyelesaikan notebook, pembaca mampu:

1. Menurunkan dan mengimplementasikan histogram diskret, PDF empiris, dan CDF beserta efek jumlah bin.
2. Menilai apakah ukuran kontras global relevan terhadap tugas deteksi lokal.
3. Menurunkan fungsi transformasi intensitas dan batas informasi yang dapat dipulihkan.
4. Menjelaskan hubungan turunan fungsi transformasi dengan amplifikasi noise.
5. Mengimplementasikan histogram equalization dari CDF dan membandingkannya dengan pustaka.
6. Menguji batas CLAHE, konversi warna, dan histogram matching.
7. Mengisolasi efek iluminasi, offset, dan noise pada pengukuran radiometrik.
8. Menyusun evaluasi yang dapat direproduksi, lengkap dengan batas klaim.

**Pertanyaan inti:** Jika sebuah citra tampak lebih kontras setelah CLAHE, apakah nilai fisik piksel menjadi lebih dapat dipercaya? Mengapa *dua citra dengan histogram identik* bisa berisi objek yang sama sekali berbeda?

## 1. Persiapan dan kontrak numerik

Kita bekerja dengan array intensitas skala `[0,1]` bila tidak dinyatakan lain. Ini **bukan** berarti semua angka merepresentasikan radiansi atau reflektansi linear. Untuk citra contoh `uint8`, normalisasi ke float semata menormalkan **nilai kode** sensor/berkas.

Fungsi `tampilkan` menggunakan skala warna eksplisit, sehingga perubahan otomatis pada batas warna tidak salah dibaca sebagai perbaikan citra. Gunakan *fixed color limits* saat membandingkan amplitudo fisik.

In [ ]:
import platform, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from skimage import data, exposure, color, img_as_float
from skimage.transform import resize
from sklearn.metrics import roc_auc_score
import skimage, scipy, sklearn

SEED = 20261112
rng = np.random.default_rng(SEED)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': False, 'font.size': 10})

def tampilkan(ax, citra, judul, lo=0, hi=1, cmap='gray'):
    ax.imshow(citra, cmap=cmap, vmin=lo, vmax=hi)
    ax.set_title(judul)
    ax.axis('off')

def hist_cdf(x, nbin=64, rentang=(0, 1)):
    h, edges = np.histogram(np.asarray(x).ravel(), bins=nbin, range=rentang)
    p = h / h.sum()                  # probabilitas satu bin, BUKAN densitas per intensitas
    cdf = np.cumsum(p)
    centers = 0.5 * (edges[1:] + edges[:-1])
    return centers, h, p, cdf

print('Python',platform.python_version(), '| NumPy',np.__version__, '| scikit-image',skimage.__version__)
assert np.isclose(hist_cdf(np.linspace(0, 1, 100))[2].sum(), 1)

## 2. Histogram sebagai estimasi distribusi

Untuk citra diskret $I[m,n]$ dengan $M\times N$ piksel dan $L$ level kode, histogram tingkat ke-$k$ adalah

$$
h[k]=\sum_{m=0}^{M-1}\sum_{n=0}^{N-1}\mathbf{1}\{I[m,n]=k\},
\qquad k=0,\dots,L-1.
$$

Dengan $N_p=MN$, probabilitas empiris $p[k]=h[k]/N_p$, sehingga $\sum_k p[k]=1$. Distribusi kumulatif adalah $F[k]=\sum_{j=0}^{k}p[j]$. Untuk citra kontinu numerik yang dikelompokkan ke bin lebar $\Delta$, $p[k]$ adalah **massa probabilitas** di dalam bin; densitas probabilitas diperkirakan $p[k]/\Delta$.

**Apa yang hilang?** Histogram tidak menyimpan koordinat piksel, hubungan ketetanggaan, struktur tepi, orientasi, atau topologi objek. Perubahan urutan piksel mempertahankan histogram, tetapi dapat merusak seluruh makna citra.

In [ ]:
base = img_as_float(data.camera())
permutasi = rng.permutation(base.size).reshape(base.shape)
teracak = base.ravel()[permutasi]
fig, ax=plt.subplots(2, 2, figsize=(10, 7))
tampilkan(ax[0,0],base,'Citra asli')
tampilkan(ax[0,1],teracak,'Piksel diacak (histogram sama)')
for im,lab in [(base,'Asli'),(teracak,'Diacak')]:
    x,_,p,F=hist_cdf(im,64)
    ax[1,0].plot(x,p,label=lab)
    ax[1,1].plot(x,F,label=lab)
ax[1,0].set(xlabel='Intensitas (kode [0,1])',ylabel='Massa probabilitas per bin',title='Histogram')
ax[1,1].set(xlabel='Intensitas (kode [0,1])',ylabel='CDF',title='Distribusi kumulatif')
for a in ax[1]: a.legend(); a.grid(alpha=.2)
plt.tight_layout();plt.show()
assert np.array_equal(np.sort(base.ravel()),np.sort(teracak.ravel()))
print('Pemeriksaan: himpunan nilai dan histogram persis sama; struktur spasial berbeda.')

### Eksperimen 2A — Mengapa jumlah bin mengubah kesan?

Nilai histogram dipengaruhi pemilihan lebar bin. Puncak distribusi dapat bergeser atau tampak melebar hanya karena binning. Fenomena ini krusial saat membandingkan kamera dengan *bit depth* berbeda atau mendeteksi populasi kecil suatu kelas intensitas.

Jangan mencampurkan `numpy.histogram(density=True)` (densitas) dan probabilitas per bin; integral densitas terhadap intensitas, bukan jumlah mentah ketinggian bin, bernilai satu.

In [ ]:
fig, axs=plt.subplots(1,3,figsize=(13,3.1))
for ax,b in zip(axs,[16,64,256]):
    centers,h,p,F=hist_cdf(base,nbin=b)
    ax.bar(centers,p,width=1/b,color='0.3')
    ax.set(title=f'{b} bin | maksimum massa = {p.max():.3f}',xlabel='Intensitas kode',ylabel='Massa probabilitas/bin',xlim=(0,1))
plt.tight_layout();plt.show()

## 3. Apa yang dimaksud kontras?

Tidak ada satu angka kontras yang sesuai untuk semua citra dan tugas. Beberapa definisi klasik:

**Kontras Michelson** pada pola periodik terang–gelap,
$$C_M=\frac{I_{\max}-I_{\min}}{I_{\max}+I_{\min}}.$$

**Kontras Weber** untuk target kecil di atas latar yang mendekati seragam,
$$C_W=\frac{I_t-I_b}{I_b},\quad I_b>0.$$

**Kontras RMS** untuk wilayah dengan intensitas $I_i$,
$$C_{\rm RMS}=\sqrt{\frac{1}{N}\sum_i (I_i-\bar I)^2}.$$

Kontras RMS mempunyai satuan yang sama dengan intensitas (jika input masih satuan fisik). Bila dinormalisasi oleh $\bar I$, muncul koefisien variasi, tetapi langkah tersebut sensitif pada mean yang mendekati nol.

Rentang persentil $P_{98}-P_2$ sering berguna untuk gambaran penyebaran global, **tetapi tidak menyatakan kemampuan mendeteksi objek**. Semakin lebar rentang histogram, bukan berarti semakin besar informasi tentang suatu cacat atau parameter fisis.

In [ ]:
# Dua pola kontras yang memiliki dinamika statistik berbeda
xx=np.linspace(0,1,512,endpoint=False)
berkala=.5+.12*np.cos(2*np.pi*7*xx)
patch=np.full(512,.5);patch[216:296]=.62

def michelson(x):
    return (x.max()-x.min())/(x.max()+x.min())

def rms(x): return float(np.std(x,ddof=0))

fig,ax=plt.subplots(1,2,figsize=(11,3))
for a,x,lab in zip(ax,[berkala,patch],['Pola periodik','Objek kecil pada latar seragam']):
    a.plot(np.arange(512),x);a.set_ylim(.32,.72)
    a.set(title=f'{lab}\nMichelson={michelson(x):.3f}; RMS={rms(x):.3f}',xlabel='Posisi piksel',ylabel='Intensitas kode')
plt.tight_layout();plt.show()
print('Weber target patch terhadap latar:',(.62-.5)/.5)

## 4. Transformasi intensitas: apa yang dapat dan tidak dapat dilakukan?

Transformasi titik bekerja per piksel: $J[m,n]=g(I[m,n])$.

1. Jika $g$ monoton ketat dan diketahui, urutan intensitas dapat dipertahankan, serta secara matematis invers tersedia pada rentang yang tidak terpotong.
2. Bila $g$ memetakan banyak nilai ke satu nilai akibat *clipping*, pembulatan, atau kuantisasi, informasi intensitas hilang dan tidak bisa dipulihkan unik.
3. Transformasi nonlinier biasanya mengubah skala noise. Dalam pendekatan perturbasi kecil $I=\mu+\epsilon$ dengan $E[\epsilon]=0$ dan varians $\sigma^2$,
$$g(\mu+\epsilon)\approx g(\mu)+g'(\mu)\epsilon,\qquad \operatorname{Var}[g(I)]\approx [g'(\mu)]^2\sigma^2.$$
4. Pada orde kedua, $E[g(I)]\approx g(\mu)+\tfrac12g''(\mu)\sigma^2$: transformasi juga dapat menghasilkan *bias*.

Karena itu, menaikkan rentang intensitas bukanlah prosedur untuk menambah foton yang telah diterima sensor. **Kontras tampilan ≠ rasio signal-to-noise yang lebih baik ≠ akurasi radiometrik.**

In [ ]:
# Periksa pendekatan propagasi noise dengan simulasi Monte Carlo
rng_noise=np.random.default_rng(SEED+1)
mu=.22; sigma=.008
eps=rng_noise.normal(0,sigma,120_000)
for gamma in [.5,1.,2.]:
    g=lambda x: x**gamma
    observasi=g(np.clip(mu+eps,0,1))
    pred_std=abs(gamma*mu**(gamma-1))*sigma
    print(f'gamma={gamma:3.1f}: simpangan baku empiris={np.std(observasi):.6f}, aproksimasi delta={pred_std:.6f}, bias={np.mean(observasi)-g(mu):+.6f}')

## 5. Contrast stretching dan normalisasi robust

Rescaling linier dari interval masukan $[a,b]$ ke $[0,1]$:

$$g(x)=\operatorname{clip}\left(\frac{x-a}{b-a},0,1\right),\qquad b>a.$$

Jika $a=\min(I)$ dan $b=\max(I)$, satu *hot pixel* dapat menentukan seluruh rentang. Alternatifnya $a=P_{2}$, $b=P_{98}$ menurunkan sensitivitas terhadap outlier tetapi **menyaturasi** 4% piksel pada kasus tanpa ties. Persentase yang benar-benar terpotong bergantung distribusi dan nilai sama.

Dalam pipeline ilmiah, bedakan dua operasi:

- **Transformasi untuk visualisasi**, diperbolehkan mengubah tampilan agar pola terlihat.
- **Kalibrasi untuk pengukuran**, harus mengikuti model respons sensor dan kondisi akuisisi; peregangan persentil bukan pengganti kalibrasi.

Untuk *machine learning*, fitting normalisasi dengan statistik gabungan train dan test adalah bentuk kebocoran informasi. Normalisasi per-citra boleh digunakan jika ditetapkan sebelum evaluasi dan sesuai dengan protokol, namun dapat menghapus informasi absolut yang justru penting untuk label.

In [ ]:
rng_robust=np.random.default_rng(SEED+2)
contoh=np.clip(.30+.04*rng_robust.normal(size=(160,160)),0,1)
contoh[4,4]=1.0; contoh[7,7]=0.
a2,b98=np.percentile(contoh,[2,98])
stretch_minmax=exposure.rescale_intensity(contoh,in_range='image',out_range=(0.,1.))
stretch_robust=np.clip((contoh-a2)/(b98-a2),0,1)
fig,axs=plt.subplots(1,3,figsize=(10.5,3.1))
for a,arr,judul in zip(axs,[contoh,stretch_minmax,stretch_robust],['Asli + 2 outlier','Min–maks','P2–P98 (clipping)']):
    tampilkan(a,arr,judul)
plt.tight_layout();plt.show()
print('Persentil P2 / P98:',round(a2,4),round(b98,4))
print('Fraksi terpotong bawah/atas:',np.mean(contoh<a2),np.mean(contoh>b98))

## 6. Transformasi gamma, logaritmik, dan makna linearitas

Untuk intensitas normalisasi **linear** $r\in[0,1]$, keluarga fungsi pangkat adalah $s=r^\gamma$ dengan $\gamma>0$. Nilai $\gamma<1$ mencerahkan bayangan; $\gamma>1$ meredamnya. Penyesuaian logaritmik misalnya $s=\log(1+\alpha r)/\log(1+\alpha)$, dengan $\alpha>0$.

**Peringatan:** data JPEG/sRGB umumnya sudah melalui kurva transfer nonlinier. Menerapkan gamma ke kode sRGB tidak sama dengan mengubah paparan optik. Tidak sah membandingkan angka dari pipeline berbeda seolah-olah semuanya mencerminkan irradiance. Notebook 01 membedakan sinyal linear sensor dari kode tampilan.

Untuk menilai apakah transformasi sesuai, tentukan terlebih dahulu *apa yang dipertahankan*: pembacaan fisik, persepsi, atau fitur untuk algoritma downstream.

In [ ]:
ruang=np.linspace(0,1,500)
fig,ax=plt.subplots(1,2,figsize=(11,3.3))
for gamma in [.4,.7,1.,1.6,2.2]:
    ax[0].plot(ruang,ruang**gamma,label=rf'$\gamma={gamma}$')
ax[0].plot(ruang,ruang,ls='--',color='0.5',alpha=.5)
ax[0].set(xlabel='Intensitas masukan',ylabel='Intensitas keluaran',title='Kurva transfer gamma');ax[0].legend(ncol=2)
redup=np.clip(.36*base+.07,0,1) # degradasi pada domain kode: hanya simulasi ilustratif
for g in [.55,1.,1.8]:
    ax[1].hist(redup.ravel()**g,bins=64,range=(0,1),histtype='step',label=f'gamma {g}')
ax[1].set(xlabel='Nilai kode keluaran',ylabel='Frekuensi piksel',title='Histogram setelah fungsi transfer');ax[1].legend()
for a in ax:a.grid(alpha=.2)
plt.tight_layout();plt.show()

## 7. Histogram equalization: derivasi dari transformasi probabilitas

Untuk variabel intensitas kontinu $R$ dengan PDF $p_R(r)$, fungsi sebaran kumulatif adalah

$$F_R(r)=\int_{-\infty}^{r}p_R(u)\,du.$$

Definisikan $S=T(R)=F_R(R)$. Untuk $F_R$ kontinu dan distribusi yang memenuhi syarat transformasi probabilitas, $S$ mengikuti distribusi seragam pada $[0,1]$. Jika $T$ dapat didiferensiasi dan monoton ketat pada daerah dengan $p_R>0$,

$$p_S(s)=p_R(r)\left|\frac{dr}{ds}\right|=\frac{p_R(r)}{dF_R/dr}=1.$$

Pada citra digital, kita tidak memiliki distribusi kontinu sempurna. Definisikan
$$T[k]=(L-1)\sum_{j=0}^{k}p[j],$$
kemudian lakukan pembulatan ke level diskret. **CDF bertangga, nilai berulang, dan jumlah level terbatas** menyebabkan histogram keluaran tidak harus seragam. Normalisasi menggunakan CDF minimum positif merupakan varian untuk memetakan level minimum yang teramati ke nol; implementasi perpustakaan lain dapat memakai interpolasi CDF berbeda.

**Inti:** Histogram equalization menyusun ulang distribusi nilai global, bukan memulihkan tekstur yang hilang atau memperbaiki model iluminasi.

In [ ]:
# Equalization diskret dari dasar (uint8) tanpa asumsi hasilnya identik dengan implementasi skimage
u8=data.camera().astype(np.uint8)
def equalize_u8_manual(a):
    assert a.dtype==np.uint8
    h=np.bincount(a.ravel(),minlength=256)
    F=np.cumsum(h)/a.size
    first=F[np.flatnonzero(h)[0]]
    if np.isclose(first,1):
        return np.zeros_like(a), np.zeros(256,dtype=np.uint8)
    lookup=np.rint(255*np.clip((F-first)/(1-first),0,1)).astype(np.uint8)
    return lookup[a],lookup
manual,lut=equalize_u8_manual(u8)
lib=exposure.equalize_hist(u8)
fig,ax=plt.subplots(2,3,figsize=(11,6.4))
for a,x,title in zip(ax[0],[u8/255,manual/255,lib],['Asli','Equalization manual','scikit-image']):tampilkan(a,x,title)
ax[1,0].plot(np.arange(256),lut);ax[1,0].set(title='LUT $T[k]$',xlabel='Level kode masukan',ylabel='Level kode keluaran')
for a,x,t in [(ax[1,1],u8,'Asli'),(ax[1,2],manual,'Manual')]:
    a.hist(x.ravel(),bins=np.arange(257)-.5,color='0.3');a.set(title=f'Histogram {t}',xlabel='Level kode',ylabel='Frekuensi')
plt.tight_layout();plt.show()
assert np.all(np.diff(lut.astype(int))>=0), 'LUT harus monoton tak-menurun'
print('Jumlah level unik masukan:',len(np.unique(u8)),'keluaran:',len(np.unique(manual)))
print('Perbedaan manual vs scikit-image (rata-rata kode):',round(float(np.mean(np.abs(manual/255-lib))),5))

### 7A. Kasus degenerat: hanya dua level

Citra biner tidak dapat diubah menjadi citra dengan 256 informasi intensitas berbeda hanya dengan histogram equalization. Semua piksel yang memiliki nilai asal sama tetap memiliki nilai transformasi sama pada pemetaan titik deterministik. Dengan kata lain, memperluas kontras tidak menciptakan *grayscale detail* yang sebelumnya tidak terukur.

In [ ]:
biner=np.zeros((90,150),dtype=np.uint8);biner[:,75:]=150
b_eq,_=equalize_u8_manual(biner)
fig,axs=plt.subplots(1,2,figsize=(8,2.6))
tampilkan(axs[0],biner/255,'Dua level awal')
tampilkan(axs[1],b_eq/255,'Hasil equalization manual')
plt.tight_layout();plt.show()
print('Jumlah level awal/akhir:',np.unique(biner).size,np.unique(b_eq).size)
assert np.unique(b_eq).size<=np.unique(biner).size

## 8. CLAHE: pemetaan lokal dan risiko artefak

*Contrast Limited Adaptive Histogram Equalization* (CLAHE) membagi citra menjadi wilayah lokal, membentuk histogram tiap wilayah, membatasi puncak hitungan histogram dengan parameter *clip limit*, lalu menginterpolasi pemetaan antartile untuk mengurangi diskontinuitas batas. Beberapa implementasi mendistribusikan kembali hitungan yang dipotong ke seluruh bin. Definisi numerik `clip_limit` bergantung pada implementasi; jangan membandingkan nilainya antarpustaka tanpa membaca dokumentasi.

**Manfaat:** pola dengan kontras rendah pada wilayah tertentu lebih mudah terlihat. **Risiko:** noise, pola sensor, dan struktur iluminasi lokal juga dapat ditonjolkan. Ini bukan transformasi radiometrik terkalibrasi. Menyesuaikan CLAHE pada gambar uji menggunakan label uji untuk memilih parameter adalah bentuk *test-set tuning*.

Eksperimen berikut memperlihatkan bahwa parameter yang semakin agresif tidak selalu menghasilkan informasi yang lebih baik.

In [ ]:
redup = np.clip(.35*base+.08,0,1)
redup_noisy = np.clip(redup+rng.normal(0,.018,base.shape),0,1)
opsi = [('Redup + noise',redup_noisy),
        ('HE global',exposure.equalize_hist(redup_noisy)),
        ('CLAHE 0.005',exposure.equalize_adapthist(redup_noisy,clip_limit=.005)),
        ('CLAHE 0.02',exposure.equalize_adapthist(redup_noisy,clip_limit=.02)),
        ('CLAHE 0.10',exposure.equalize_adapthist(redup_noisy,clip_limit=.10))]
fig,ax=plt.subplots(2,5,figsize=(15,5.7))
for j,(judul,arr) in enumerate(opsi):
    tampilkan(ax[0,j],arr,judul)
    ax[1,j].hist(arr.ravel(),bins=64,range=(0,1),color='0.3')
    ax[1,j].set(xlim=(0,1),title='Distribusi keluaran',xlabel='Kode piksel')
    if j==0:ax[1,j].set_ylabel('Frekuensi')
plt.tight_layout();plt.show()

## 9. Histogram matching dan pemilihan *domain*

Pada *histogram specification* atau *matching*, transformasi intensitas dicari agar CDF citra sumber mendekati CDF citra acuan. Untuk dua CDF kontinu monoton ketat, secara ideal transformasinya

$$
T(r)=F_{\mathrm{ref}}^{-1}\big(F_{\mathrm{src}}(r)\big).
$$

Metode ini berguna untuk konsistensi tampilan, tetapi **tidak menyamakan kondisi fisik akuisisi**; kamera, respons spektral, paparan, atau saturasi tidak hilang hanya karena histogramnya serupa. Pemetaan kuantil pada citra diskret memerlukan interpolasi dan perlakuan khusus terhadap nilai identik. Histogram matching dapat menyesatkan jika distribusi isi citra (proporsi latar dan objek) berbeda: algoritma mungkin memaksa citra obyek yang berbeda agar statistik globalnya mirip.

In [ ]:
src=np.clip(.40*base+.07,0,1)
ref=img_as_float(data.moon())
matched=exposure.match_histograms(src,ref)
fig,ax=plt.subplots(2,3,figsize=(12,6.6))
for a,x,judul in zip(ax[0],[src,ref,matched],['Sumber','Acuan (isi berbeda)','Sumber sesudah matching']):
    tampilkan(a,x,judul)
for a,x,judul in zip(ax[1],[src,ref,matched],['Sumber','Acuan','Matching']):
    bins,pedges=np.histogram(x,bins=128,range=(0,1),density=True)
    a.plot((pedges[:-1]+pedges[1:])/2,np.cumsum(bins)/np.sum(bins))
    a.set(xlim=(0,1),ylim=(0,1.01),title='CDF '+judul,xlabel='Kode intensitas',ylabel='CDF')
    a.grid(alpha=.2)
plt.tight_layout();plt.show()

## 10. Citra berwarna: jangan lakukan equalization RGB secara membabi buta

Warna RGB merupakan **representasi vektor**, bukan tiga besaran yang sepenuhnya bebas dari pengaruh persepsi dan proses tampilan. Melakukan equalization secara terpisah pada R, G, dan B dapat mengubah hubungan antarkanal, sehingga rona (*hue*) objek bergeser. Salah satu alternatif untuk tujuan **visualisasi** ialah mengolah komponen luminans/terang pada ruang warna yang memisahkan informasi kecerahan dari kromatisitas, misalnya ruang Lab atau HSV.

Perhatian: komponen `L*` pada CIE Lab adalah *lightness* perseptual, **bukan** luminansi fisik linear. Konversi dari sRGB memerlukan transformasi transfer yang sesuai, dan perubahan pada kanal L* **tidak** otomatis memberikan kalibrasi kolorimetri. Dalam pengukuran warna, prosedur referensi, *white balance*, kurva respons, kondisi iluminasi, dan ketidakpastian tetap diperlukan.

In [ ]:
rgb=img_as_float(data.astronaut())
rgb_kusam=np.clip(.48*rgb+.04,0,1)
rgb_per_channel=np.stack([exposure.equalize_hist(rgb_kusam[:,:,j]) for j in range(3)],axis=2)
lab=color.rgb2lab(rgb_kusam)
lab_out=lab.copy()
lab_out[:,:,0]=100*exposure.equalize_adapthist(lab[:,:,0]/100,clip_limit=.012)
rgb_lab=np.clip(color.lab2rgb(lab_out),0,1)
fig,ax=plt.subplots(1,3,figsize=(12.8,3.8))
for a,x,jud in zip(ax,[rgb_kusam,rgb_per_channel,rgb_lab],
                   ['RGB awal (simulasi kusam)','Equalize setiap kanal RGB','CLAHE hanya lightness Lab']):
    a.imshow(x);a.set_title(jud);a.axis('off')
plt.tight_layout();plt.show()
# Selisih kromatisitas hanya sebagai indikator; ini BUKAN studi ΔE terkalibrasi.
print('Perubahan RGB rata-rata, per-kanal vs Lab:',
      np.mean(np.abs(rgb_per_channel-rgb_kusam)),np.mean(np.abs(rgb_lab-rgb_kusam)))

## 11. Dari estetika ke metrologi: model *dark–flat field*

Bayangkan inspeksi lembar material dengan satu kamera. Nilai objek yang sama bisa berbeda antarpiksel karena gradien iluminasi, *vignetting*, variasi sensitivitas piksel, dan *offset* elektronik. Model sederhana pengukuran citra pada piksel $(m,n)$ adalah

$$
D_{mn}=b_{mn}+a_{mn}R_{mn}+\epsilon_{mn},
$$

di mana $R_{mn}$ adalah besaran relatif yang ingin diukur (misalnya reflektansi ternormalisasi di bawah asumsi tertentu), $a_{mn}$ adalah gain efektif termasuk pencahayaan dan respons sensor, $b_{mn}$ adalah offset, dan $\epsilon_{mn}$ adalah noise. Ini model **linear teridealisasi**: sensor harus bekerja di rentang tidak tersaturasi, kanal dan paparan konsisten, geometri serta iluminasi stabil, dan berkas yang diproses belum ditransformasikan secara nonlinier oleh *tone mapping*.

Jika citra gelap (*dark frame*) mengukur $D^{(0)}\approx b$, sedangkan citra bidang putih homogen (*flat frame*, nilai referensi $R_{\rm ref}$) mengukur $D^{(f)}\approx b+aR_{\rm ref}$, maka

$$
\widehat R_{mn}=R_{\rm ref}\frac{D_{mn}-D^{(0)}_{mn}}{D^{(f)}_{mn}-D^{(0)}_{mn}}.
$$

Dalam praktik, gunakan sejumlah *dark/flat frames* untuk menekan ketidakpastian, dan lakukan masking bagi piksel jenuh, rusak, atau penyebut yang terlalu kecil. Histogram equalization tidak menyediakan pengukuran $a$ dan $b$, sehingga tidak dapat menggantikan proses kalibrasi ini.

### 11A. Propagasi ketidakpastian

Dengan $N=D-D_0$, $Q=D_f-D_0$, dan $\widehat R=R_{\rm ref}N/Q$, pendekatan linear memberikan

$$
\operatorname{Var}(\widehat R)\approx \mathbf J\,\boldsymbol\Sigma\,\mathbf J^\mathsf T,
\qquad \mathbf J=\left[\frac{R_{\rm ref}}{Q},\ -\frac{R_{\rm ref}}{Q}+\frac{R_{\rm ref}N}{Q^2},\ -\frac{R_{\rm ref}N}{Q^2}\right],
$$

dengan urutan variabel $(D,D_0,D_f)$ dan $\boldsymbol\Sigma$ matriks kovariansnya. Rumus ini **tidak** menyatakan bahwa citra dark dan flat bebas secara otomatis; ketergantungan pengukuran dan proses averaging harus diperhitungkan. Penyebut kecil membuat sensitivitas estimasi membesar. Bahkan jika rata-rata benar, hasil tiap piksel tetap memiliki ketidakpastian.

In [ ]:
# Bukti singkat: koreksi dark-flat tidak identik dengan contrast stretching.
H,W=160,208
y,x=np.mgrid[0:H,0:W]
R_ref=1.
R_obj=.56+.012*np.cos(2*np.pi*x/43)*np.cos(2*np.pi*y/33)
for cx,cy,r in [(55,55,13),(143,86,16),(104,124,11)]:
    R_obj[(x-cx)**2+(y-cy)**2<r**2]-=.14
R_obj=np.clip(R_obj,0,1)
gain=.50*(.62+.55*x/(W-1))*(1-.26*((x-W/2)/(W/2))**2)
offset=.035+.015*(y/(H-1))
rng_cal=np.random.default_rng(SEED+9)
noisy=lambda clean: clean+rng_cal.normal(0,.008,clean.shape)
D=noisy(offset+gain*R_obj)
Dark=np.mean([noisy(offset) for _ in range(12)],axis=0)
Flat=np.mean([noisy(offset+gain) for _ in range(12)],axis=0)
R_est=(D-Dark)/(Flat-Dark)
D_stretch=np.clip((D-np.percentile(D,2))/(np.percentile(D,98)-np.percentile(D,2)),0,1)
D_eq=exposure.equalize_hist(np.clip(D,0,1))
D_clahe=exposure.equalize_adapthist(np.clip(D,0,1),clip_limit=.015)
fig,ax=plt.subplots(2,3,figsize=(12,7.3))
for a,im,judul in zip(ax.flat,[R_obj,D,D_stretch,D_eq,D_clahe,R_est],
        ['Referensi $R$ (sintetis)','Pengukuran mentah $D$','Contrast stretching P2–P98','Equalization global','CLAHE 0.015','Dark–flat correction']):
    tampilkan(a,im,judul)
plt.tight_layout();plt.show()
print('MAE (unit reflektansi simulasi) -- dark-flat:',np.mean(np.abs(R_est-R_obj)))
print('MAE angka kode vs referensi -- CLAHE (bukan metrik kalibrasi yang sebanding):',np.mean(np.abs(D_clahe-R_obj)))
print('Rentang penyebut flat-dark:',float(np.min(Flat-Dark)),float(np.max(Flat-Dark)))

### 11B. Propagasi noise: uji simulasi terhadap metode delta

Kita mengisolasi satu piksel dengan model normal sederhana agar dapat membandingkan simpangan baku hasil Monte Carlo terhadap aproksimasi orde pertama. Ini bukan validasi untuk semua model noise kamera: dekat saturasi, penyebut mendekati nol, dan noise non-Gaussian, aproksimasi dapat gagal.

In [ ]:
ran=np.random.default_rng(SEED+10)
R0=.55; a0=.45; b0=.04; sigma_d=.009; sigma_dark=.003; sigma_flat=.003
Nsim=80_000
obs=ran.normal(b0+a0*R0,sigma_d,Nsim)
dark_sample=ran.normal(b0,sigma_dark,Nsim)
flat_sample=ran.normal(b0+a0,sigma_flat,Nsim)
rhat=(obs-dark_sample)/(flat_sample-dark_sample)
q=a0;n=a0*R0
J=np.array([1/q, -1/q+n/q**2, -n/q**2])
sigma_est=np.sqrt(np.sum((J*np.array([sigma_d,sigma_dark,sigma_flat]))**2))
print(f'R0={R0:.3f} | mean estimasi={rhat.mean():.5f} | std Monte Carlo={rhat.std():.5f} | std delta={sigma_est:.5f}')
assert abs(rhat.std()-sigma_est)/sigma_est<.03

## 12. Studi kasus integratif — Inspeksi permukaan dengan iluminasi tidak seragam

**Pertanyaan engineering.** Sistem kamera harus menandai cacat gelap di permukaan material. Intensitas citra bergantung pada variasi reflektansi benda, iluminasi, offset, dan noise. Seorang operator mengusulkan CLAHE karena terlihat lebih kontras, sedangkan insinyur pengukuran mengusulkan kalibrasi dark–flat. Mana yang lebih konsisten untuk deteksi dalam pengaturan ini?

Kita membuat **dataset sintetis** dengan *ground truth* mask lingkaran cacat, satu citra untuk setiap spesimen, dan kondisi iluminasi yang berubah. Data tidak berasal dari instrumen nyata; hasil tidak boleh dijadikan klaim performa pada pabrik. Pembagian di tingkat **spesimen**, bukan piksel, untuk menghindari kebocoran antartrain dan test.

**Protokol:** gunakan 8 spesimen validasi untuk memilih ambang tiap metode, lalu 12 spesimen uji yang terpisah. Semua pengolahan bersifat tanpa pembelajaran, tetapi pemilihan ambang tetap merupakan tahap *fitting*. Penggunaan label test untuk memilih threshold dilarang.

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score

HEIGHT,WIDTH=128,160
YY,XX=np.mgrid[:HEIGHT,:WIDTH]

def spesimen(seed):
    rr=np.random.default_rng(seed)
    ys,xs=YY,XX
    # Komponen material: latar bertekstur + cacat bulat berlabel.
    textura=ndi.gaussian_filter(rr.normal(size=(HEIGHT,WIDTH)),sigma=3)
    textura= .015*textura/(np.std(textura)+1e-12)
    refl=np.clip(.55+textura,0,1)
    mask=np.zeros((HEIGHT,WIDTH),dtype=bool)
    for k in range(3):
        cy=int(rr.integers(16,HEIGHT-16));cx=int(rr.integers(16,WIDTH-16));r=int(rr.integers(5,10))
        mask |= (xs-cx)**2+(ys-cy)**2<=r*r
    refl=np.clip(refl-.15*mask,0,1)
    grad=rr.uniform(.38,.60)+rr.uniform(.24,.42)*(xs/(WIDTH-1))
    vignette=1-rr.uniform(.20,.38)*((xs-WIDTH/2)/(WIDTH/2))**2
    a=np.clip(grad*vignette,.12,1.)
    b=.03+rr.uniform(0,.025)*(ys/(HEIGHT-1))
    sd=rr.uniform(.006,.014)
    obs=np.clip(b+a*refl+rr.normal(0,sd,refl.shape),0,1)
    dk=np.mean([b+rr.normal(0,sd,refl.shape) for _ in range(10)],axis=0)
    fl=np.mean([b+a+rr.normal(0,sd,refl.shape) for _ in range(10)],axis=0)
    corrected=(obs-dk)/np.maximum(fl-dk,.08)
    # Per-scene preprocessing ditetapkan sebelum melihat label; dilakukan independen.
    hist=exposure.equalize_hist(obs)
    clahe=exposure.equalize_adapthist(obs,clip_limit=.012)
    lo,hi=np.percentile(obs,[2,98]);scale=np.clip((obs-lo)/(hi-lo),0,1)
    return {'label':mask,'refl':refl,'raw':obs,'stretch':scale,
            'hist_eq':hist,'clahe':clahe,'calibrated':corrected}

valid=[spesimen(SEED+100+i) for i in range(8)]
test=[spesimen(SEED+200+i) for i in range(12)]
metode=['raw','stretch','hist_eq','clahe','calibrated']

fig,ax=plt.subplots(2,3,figsize=(11.4,6.1))
scene=test[0]
for a,(name,img) in zip(ax.flat,[('Referensi reflektansi',scene['refl']),('Mask cacat',scene['label']),
    ('Citra mentah',scene['raw']),('Global HE',scene['hist_eq']),('CLAHE',scene['clahe']),('Dark-flat',scene['calibrated'])]):
    tampilkan(a,img,name)
plt.tight_layout();plt.show()
print('Validation scenes:',len(valid),'Test scenes:',len(test),'Defect pixel fraction:',round(float(np.mean(scene['label'])),3))

### 12A. Seleksi ambang **hanya pada validation set**

Untuk semua metode, cacat dibuat lebih gelap dibanding latar sehingga kita gunakan skor $s=-I$. Ambang dipilih guna memaksimumkan F1 pada piksel dari spesimen validasi. Ini adalah simplifikasi algoritma segmentasi intensitas; perangkat riil mungkin membutuhkan fitur spasial dan segmentasi lebih canggih.

Ingat bahwa $\mathrm{F1}=2PR/(P+R)$, dengan $P=TP/(TP+FP)$ dan $R=TP/(TP+FN)$. Karena cacat jarang, akurasi biasa dapat menyesatkan. Kita melaporkan F1, precision, recall, dan AUROC; nilai AUROC pada tugas yang sangat tidak seimbang juga perlu disertai evaluasi precision–recall bila diaplikasikan lebih lanjut.

In [ ]:
def pilih_ambang(kasus,field):
    yy=np.concatenate([s['label'].ravel() for s in kasus]).astype(bool)
    zz=np.concatenate([-s[field].ravel() for s in kasus])
    # kandidat kuantil ditentukan hanya dari data validasi
    kandidat=np.unique(np.quantile(zz,np.linspace(.65,.995,160)))
    hasil=np.array([f1_score(yy,zz>=t,zero_division=0) for t in kandidat])
    idx=np.argmax(hasil)
    return float(kandidat[idx]),float(hasil[idx])

thresholds={}
for field in metode:
    t,best=pilih_ambang(valid,field)
    thresholds[field]=t
    print(f'{field:11s} | validasi: threshold skor={t: .4f} | F1={best:.3f}')

### 12B. Evaluasi pada spesimen uji dan ketidakpastian antarspesimen

Setelah ambang dipilih, **jangan mengubahnya menggunakan data uji**. Kita hitung metrik setiap spesimen, kemudian merangkum nilai rata-ratanya. Sebagai gambaran ketidakpastian sampling antarspesimen, lakukan *bootstrap* pada unit spesimen, bukan memperlakukan puluhan ribu piksel bertetangga sebagai observasi bebas.

Interval bootstrap persentil di bawah adalah gambaran variasi dalam distribusi spesimen sintetis ini, **bukan** interval ketidakpastian untuk kinerja pada semua material dan kamera di dunia nyata. Untuk inferensi pada studi sungguhan, rancangan pengambilan sampel dan jumlah *batch*, perangkat, serta lokasi harus diperhitungkan.

In [ ]:
from sklearn.metrics import precision_score,recall_score
records=[]
for field in metode:
    for i,s in enumerate(test):
        gt=s['label'].ravel()
        score=(-s[field]).ravel()
        pred=score>=thresholds[field]
        records.append(dict(metode=field,scene=i,
            precision=precision_score(gt,pred,zero_division=0),
            recall=recall_score(gt,pred,zero_division=0),
            F1=f1_score(gt,pred,zero_division=0),
            AUROC=roc_auc_score(gt,score)))
res=pd.DataFrame(records)
rng_ci=np.random.default_rng(SEED+500)
summary=[]
for field,group in res.groupby('metode',sort=False):
    f1v=group['F1'].to_numpy()
    draws=rng_ci.choice(f1v,size=(2000,len(f1v)),replace=True).mean(axis=1)
    summary.append(dict(metode=field,F1_mean=f1v.mean(),CI95_lower=np.quantile(draws,.025),
            CI95_upper=np.quantile(draws,.975),precision_mean=group.precision.mean(),
            recall_mean=group.recall.mean(),AUROC_mean=group.AUROC.mean()))
tabel=pd.DataFrame(summary).set_index('metode').round(3)
print('Evaluasi pada 12 spesimen uji independen dari set validasi:')
print(tabel.to_string())
fig,ax=plt.subplots(figsize=(9,3.8))
for i,row in enumerate(summary):
    v=row['F1_mean'];ax.errorbar(v,i,xerr=[[v-row['CI95_lower']],[row['CI95_upper']-v]],fmt='o',capsize=4)
ax.set(yticks=range(len(summary)),yticklabels=[r['metode'] for r in summary],xlim=(0,1),xlabel='F1 rata-rata per spesimen (bootstrap 95%)',title='Perbandingan metode pada data simulasi')
ax.grid(alpha=.2,axis='x');plt.tight_layout();plt.show()
assert len(set(res['scene']))==12 and res.shape[0]==12*len(metode)

### 12C. Lihat kegagalan: kesalahan keputusan di wilayah cacat

Metrik rata-rata menyembunyikan lokasi kegagalan. Tampilkan *false positive* dan *false negative*. Daerah gelap karena iluminasi sering tertukar dengan cacat jika hanya memakai threshold global pada citra mentah. Hasil aktual bergantung pada seed dan parameter simulasi yang ditetapkan dalam notebook.

In [ ]:
example=test[1]
fig,ax=plt.subplots(2,3,figsize=(11.5,6.8))
tampilkan(ax[0,0],example['raw'],'Pengukuran mentah')
tampilkan(ax[0,1],example['calibrated'],'Reflektansi estimasi (dark-flat)')
tampilkan(ax[0,2],example['label'],'Mask ground truth')
for j,field in enumerate(['raw','clahe','calibrated']):
    pred=(-example[field])>=thresholds[field]
    out=np.zeros((*pred.shape,3),dtype=float)
    out[...,1]=pred & example['label'] # true positive hijau
    out[...,0]=pred & ~example['label'] # false positive merah
    out[...,2]=~pred & example['label'] # false negative biru
    ax[1,j].imshow(out);ax[1,j].axis('off')
    ax[1,j].set_title(f'{field}: TP hijau, FP merah, FN biru')
plt.tight_layout();plt.show()

## 13. Audit asumsi dan interpretasi

Sebelum menyimpulkan bahwa satu metode lebih baik, jawab pertanyaan berikut:

1. **Tujuan:** apakah optimasi diarahkan pada tampilan bagi manusia, kuantifikasi reflektansi, atau deteksi cacat?
2. **Definisi referensi:** apakah intensitas berasal dari nilai kode JPEG, sinyal linear sensor, atau reflektansi terkalibrasi?
3. **Data:** apakah *dark/flat frame* disampling pada paparan dan iluminasi yang sama? Apakah ada saturasi, *bad pixels*, atau drift?
4. **Eksperimen:** apakah pemilihan hyperparameter/threshold hanya memakai validasi, lalu test benar-benar terpisah di tingkat spesimen atau batch?
5. **Evaluasi:** apakah metrik merefleksikan biaya kegagalan aplikasi? Apakah ada ketidakpastian antarspesimen dan *domain shift* yang belum diuji?
6. **Klaim:** apakah contoh sintetis hanya menyatakan kemungkinan mekanisme atau telah membuktikan keberhasilan pada instrumen sebenarnya? **Di sini hanya yang pertama.**

Khusus penggunaan machine learning, histogram, CLAHE, dan normalisasi bukan prosedur yang otomatis meningkatkan kemampuan generalisasi. Transformasi itu bisa menghapus sinyal yang relevan untuk label atau justru membuat model belajar ciri akuisisi yang tidak stabil.

## 14. Latihan bertingkat dan analisis HOTS

**Latihan 1 — Distribusi (dasar).** Untuk matriks $4\times 4$ berisi empat level $(0,1,2,3)$ dengan frekuensi $(2,6,5,3)$, hitung $h[k]$, $p[k]$, CDF, dan LUT histogram equalization pada $L=4$. Jelaskan mengapa hasil diskret tidak harus seragam.

**Latihan 2 — Noise (analitis).** Asumsikan $I=\mu+\epsilon$ dengan $\mu=0.25$ dan $\sigma=0.01$. Bandingkan pendekatan simpangan baku keluaran untuk $g(I)=I^{0.5}$, $g(I)=I$ dan $g(I)=I^2$. Turunkan fungsi $g'(\mu)$ terlebih dahulu.

**Latihan 3 — Implementasi (komputasional).** Implementasikan equalization berbasis histogram dengan NumPy tanpa `skimage.exposure.equalize_hist`. Uji citra seragam dan dua level, bukan hanya citra contoh.

**Latihan 4 — Citra berwarna (analitis-komputasional).** Bandingkan equalization RGB per kanal dengan perubahan lightness di Lab pada *region of interest* yang sama. Jelaskan apakah perubahan histogram tiap kanal cukup membuktikan bahwa rona warna tidak bergeser.

**Latihan 5 — Metrologi (HOTS).** Sebuah piksel memiliki $(D,D_0,D_f)=(0.42,0.06,0.66)$ dan masing-masing simpangan baku $(0.01,0.004,0.005)$, bebas satu sama lain. Hitung $\widehat R$, turunkan ketidakpastian dengan Jacobian, dan analisis dua skenario ketika $D_f-D_0$ turun menjadi 0.06 dengan noise yang sama: (a) $D-D_0$ tetap dan (b) $D-D_0$ ikut turun sehingga reflektansi sebenarnya tetap 0.60. Mengapa keduanya menuntut interpretasi berbeda?

**Latihan 6 — Eksperimen dan validasi (HOTS).** Ubah karakteristik pencahayaan pada *spesimen test saja* (misal lebih redup/berbentuk berbeda). Uji apakah urutan metode berdasarkan F1 tetap sama. Jelaskan perubahan domain, kemungkinan penyebabnya, dan bagaimana protokol validasi seharusnya dirancang ulang.

**Latihan 7 — Desain sistem (HOTS).** Untuk inspeksi lembaran logam dan pengukuran perubahan warna indikator, pilih pipeline yang berbeda. Jelaskan mengapa CLAHE cocok atau tidak untuk *display*, dan mengapa kalibrasi warna/radiometri tidak dapat digantikan oleh pemetaan histogram.

**Output laporan:** (i) hipotesis, (ii) diagram pipeline, (iii) model dan asumsi, (iv) parameter serta seed, (v) hasil grafik/tabel, (vi) analisis residual dan kegagalan, (vii) keterbatasan, (viii) kesimpulan sesuai kekuatan bukti.

## 15. Rangkuman dan koneksi berikutnya

1. Histogram menyatakan **distribusi kode**, bukan susunan objek atau besaran fisis yang otomatis terkalibrasi.
2. Berbagai definisi kontras bergantung tugas. Peningkatan kontras tampilan tidak selalu berarti perbaikan kemampuan deteksi atau *signal-to-noise ratio*.
3. *Contrast stretching*, gamma, equalization, dan CLAHE adalah transformasi intensitas yang mempunyai efek berbeda terhadap dinamika intensitas, noise, dan interpretasi.
4. Tidak ada pemetaan intensitas yang dapat mengembalikan informasi yang hilang akibat saturasi atau kuantisasi tanpa asumsi/informasi tambahan.
5. Pengukuran ilmiah memerlukan model sensor, referensi, dan propagasi ketidakpastian; histogram equalization tidak menggantikan kalibrasi.
6. Evaluasi yang kuat memisahkan tahap pemilihan parameter dan pengujian serta menggunakan unit eksperimental yang relevan.

**Koneksi berikutnya:** histogram dan transformasi intensitas memperdalam distribusi sinyal; filtering dan domain frekuensi memperdalam PSF/OTF; inverse problems membahas rekonstruksi di bawah noise dan informasi yang tidak lengkap; pengukuran ilmiah membahas propagasi ketidakpastian dan validasi metrologi.

**Pertanyaan penutup:** sebuah citra dapat terlihat jauh lebih tajam/kontras, tetapi mengapa insinyur belum tentu boleh menyimpulkan bahwa pembacaan instrumennya menjadi lebih akurat?

## 16. Bacaan lanjutan dan sumber

- Gonzalez, R. C. & Woods, R. E. *Digital Image Processing*, edisi ke-4, Pearson (2018), bab peningkatan citra dan histogram.
- Szeliski, R. *Computer Vision: Algorithms and Applications*, edisi ke-2, Springer (2022), bagian image formation dan image processing. https://szeliski.org/Book/
- Burger, W. & Burge, M. J. *Digital Image Processing: An Algorithmic Introduction Using Java*, Springer, edisi ke-3 (2022), bagian operasi intensitas dan histogram.
- Dokumentasi scikit-image: `skimage.exposure`, `equalize_hist`, `equalize_adapthist`, `match_histograms`. https://scikit-image.org/docs/stable/api/skimage.exposure.html
- Dokumentasi NumPy histogram: https://numpy.org/doc/stable/reference/generated/numpy.histogram.html
- JCGM 100:2008 (GUM), *Evaluation of measurement data — Guide to the expression of uncertainty in measurement*. https://www.bipm.org/en/committees/jc/jcgm/publications

**Catatan referensi:** materi ini menyusun ulang konsep fundamental dan interpretasi metode; tabel dan angka dari simulasi notebook merupakan hasil komputasi, bukan pengutipan hasil eksperimen empiris literatur.